In [1]:
import datetime
import time
import polars as pl
import pandas as pd
import re
import requests
from io import StringIO
import json


In [3]:
API_TOKEN = open(file="secrets-jretrieve", mode="r").read()
auth_url='https://service.meteoswiss.ch/auth/realms/meteoswiss.ch/protocol/openid-connect/token'
auth_data = (('grant_type', 'refresh_token'), ('client_id', 'api-token'), ('refresh_token', API_TOKEN))
# using requests
res = requests.post(url=auth_url, data=auth_data)
auth_header = 'Bearer ' + json.loads(res.text)['access_token']

jretrieve_base_url = 'https://service.meteoswiss.ch/jretrieve/api/v1/surface/nat_abbr?'

# tre200h0	1 	°C	261 	Lufttemperatur 2 m über Boden; Stundenmittel
# ure200h0	1 	%	266 	Relative Luftfeuchtigkeit 2 m über Boden; Stundenmittel
# dkl010h0	1 	°	282 	Windrichtung; Stundenmittel
# fkl010h0	1 	m/s	283 	Windgeschwindigkeit skalar; Stundenmittel
# gre000h0	1 	W/m²	269 	Globalstrahlung; Stundenmittel
# prestah0	1 	hPa	306 	Luftdruck auf Barometerhöhe (QFE); Stundenmittel
# rre150h0	1 	mm	267 	Niederschlag; Stundensumme
qry_params = 'delimiter=,&placeholder=None&locationIds=KEMKN&date=20231201000000-20231203000000&parameterShortNames=tre200h0,prestah0,ure200h0,fkl010h0,dkl010h0,rre150h0,gre000h0&measCatNr=1'
jretrieve_url = jretrieve_base_url + qry_params
res = requests.get(url=jretrieve_url, headers={'Authorization': auth_header})

df = pl.read_csv(StringIO(res.text), separator=',', null_values='None')
print(df.head())


shape: (5, 9)
┌─────────┬────────────────┬──────────┬──────────┬───┬──────────┬──────────┬──────────┬──────────┐
│ station ┆ termin         ┆ tre200h0 ┆ prestah0 ┆ … ┆ fkl010h0 ┆ dkl010h0 ┆ rre150h0 ┆ gre000h0 │
│ ---     ┆ ---            ┆ ---      ┆ ---      ┆   ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ i64     ┆ i64            ┆ str      ┆ str      ┆   ┆ f64      ┆ i64      ┆ str      ┆ str      │
╞═════════╪════════════════╪══════════╪══════════╪═══╪══════════╪══════════╪══════════╪══════════╡
│ 10769   ┆ 20231201000000 ┆ null     ┆ null     ┆ … ┆ 4.5      ┆ 117      ┆ null     ┆ null     │
│ 10769   ┆ 20231201010000 ┆ null     ┆ null     ┆ … ┆ 3.5      ┆ 92       ┆ null     ┆ null     │
│ 10769   ┆ 20231201020000 ┆ null     ┆ null     ┆ … ┆ 2.9      ┆ 99       ┆ null     ┆ null     │
│ 10769   ┆ 20231201030000 ┆ null     ┆ null     ┆ … ┆ 3.8      ┆ 124      ┆ null     ┆ null     │
│ 10769   ┆ 20231201040000 ┆ null     ┆ null     ┆ … ┆ 2.3      ┆ 135      ┆ null     ┆ null   

In [ ]:
file = "mkn_meteo_1h.parquet"
df.write_parquet(file)

df = pl.read_parquet(file)

In [ ]:
# %%



# def jretrieve_data(station, cfg=None, vars=None, category=None, duration=None, since=None, till=None, drop_null=True) -> dict:
#     """
#     Download data from DWH using jretrieve.

#     Documentation: https://object.gever.admin.ch/web/?ObjectToOpenID=%24ActaNovaDocument%7cBB0E8163-47A0-4DC2-94E2-65D1DC842A94&TenantID=180&OpenContentOfProperty=UnifiedIDocument
#     Test URL: https://service.meteoswiss.ch/jretrieve/api/v1/surface?locationIds=SMA,SCU&parameterIds=91&date=20190325000000-20190326000000&infoOptions=nat_abbr,elev&useLimitation=50

#     vars: a comma-separated list of DWH parameters

#     :return:
#     """
#     # obtain (personal) offline token at: https://service.meteoswiss.ch/api-token/
#     # store in file 'api-token' - make sure to include this in .gitignore!!
#     API_TOKEN = open(file="py/api-token", mode="r").read()
#     try:
#         # get access token (for auth_header)
#         auth_url='https://service.meteoswiss.ch/auth/realms/meteoswiss.ch/protocol/openid-connect/token'
#         auth_data = (('grant_type', 'refresh_token'), ('client_id', 'api-token'), ('refresh_token', API_TOKEN))
#         res = requests.post(url=auth_url, data=auth_data)
#         auth_header = 'Bearer ' + json.loads(res.text)['access_token']
        
#         base_url = 'https://service.meteoswiss.ch/jretrieve/api/v1'
#         base_url = base_url + '/surface/nat_abbr?delimiter=,&placeholder=None'
#         base_url += "&locationIds=%s" % station

#         urls = []
#         df = pl.DataFrame()

#         if cfg is None:
#             if vars is None:
#                 raise('"vars" not specified.')
#             if category is None:
#                 raise('"category" not specified.')
#             if duration is None:
#                 raise('"duration" not specified.')
#             else:
#                 since = (datetime.datetime.now() - datetime.timedelta(days=duration)).strftime("%Y%m%d%H%M%S")
#                 till = time.strftime("%Y%m%d%H%M%S")
#                 base_url += "&date=%s-%s" % (since, till)
#                 base_url += "&parameterShortNames=%s" % vars
#                 base_url += "&measCatNr=%s" % category
#             urls.append(base_url)
#             series = "".split(vars, sep=',')
#             labels = dict(zip(series, series))
#         else:
#             if since is None:
#                 if duration:
#                     since = (datetime.datetime.now() - datetime.timedelta(days=duration)).strftime("%Y%m%d%H%M%S")
#                 else:
#                     since = cfg[station]['since']
#             if till is None:
#                 till = time.strftime("%Y%m%d%H%M%S")
#             base_url += "&date=%s-%s" % (since, till)
            
#             params = cfg[station]['params']

#             series = [] # acronyms of var name in the context of SPICE
#             vars = []   # DWH parameter names
#             labels = []
#             for key in params.keys():
#                 url = base_url + "&parameterShortNames=%s" % params[key]['var']
#                 url += "&measCatNr=%s" % str(params[key]['cat'])
#                 urls.append(url)
#                 series.append(key)
#                 vars.append(params[key]['var'])
#                 labels.append("%s (%s)" % (params[key]['lbl'], params[key]['var']))
#             labels = dict(zip(series, labels))
#             urls = dict(zip(urls, series))

#         for url, series in urls.items():
#             print("Calling %s ..." % url)
#             t0 = time.time()
#             res = requests.get(url=url, headers={'Authorization': auth_header})

#             print("Finished downloading in %s seconds." % str(time.time() - t0))
#             if res.status_code == 200:
#                 # return res.text as Pandas dataframe, convert date/time
#                 if df.empty:
#                     # df = pd.read_csv(StringIO(res.text), na_values='None', parse_dates=['termin'], index_col=['termin'])
#                     df = pd.read_csv(StringIO(res.text), na_values='None', parse_dates=['termin'], index_col=['termin'])
#                     df.drop(columns='station', inplace=True)
#                     df.columns.values[0] = series
#                     print(f"{series}: {df.columns}")
#                     # if drop_null:
#                     #     df = df[df.columns[~df.isnull().all()]]
#                 else:
#                     df2 = pd.read_csv(StringIO(res.text), na_values='None', parse_dates=['termin'],
#                                             index_col=['termin'])
#                     df2.drop(columns='station', inplace=True)
#                     df2.columns.values[0] = series
#                     print(f"{series}: {df2.columns}")
#                     try:
#                         df = df.merge(df2, how='outer', left_index=True, right_index=True)
#                         print(f"merged: {df.columns} {len(df)} rows.")
#                     except Exception as err:
#                         print(df.info())
#                         print(df2.info())
#                         continue
#                     # if drop_null:
#                     #     df = df[df.columns[~df.isnull().all()]]
#         df.index.rename('dtm', inplace=True)
#         print(f"merged: {df.columns} {len(df)} rows.")

#         ## downcast data to reduce size of dataframe
#         for column in df:
#             if df[column].dtype == 'float64':
#                 df[column]=pd.to_numeric(df[column], downcast='float')
#             if df[column].dtype == 'int64':
#                 df[column]=pd.to_numeric(df[column], downcast='integer')
        
#         # make sure df is sorted by date
#         df.sort_index(inplace=True)

#         return {'data': df, 'labels': labels}

#     except Exception as err:
#         print(err)
